# Modwheel vs Random Matched Subsampling (Colab)

This notebook tests the next value-added question:

> Is deterministic wheel filtering doing anything more structured than simply keeping the same fraction of rows at random?

For each wheel (`mod30`, `mod210`, `mod2310`), we compare:

```text
wheel-filtered subset
vs
random subset with the same sample count
```

Dataset/workflow:
- 20 Newsgroups, 4 categories
- TF-IDF features
- LinearSVC classifier
- balanced accuracy + training time proxy

Guardrail: row-ID wheel filtering is an adapter test. Row order can encode dataset-specific artifacts, so results must be interpreted cautiously. This notebook does **not** claim quantum advantage or accuracy improvement.

In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS
from pre_oracle_filter import pre_oracle_candidates_by_name

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

## 1. Load and vectorize 20 Newsgroups

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 20

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = dataset.data
y = np.array(dataset.target)
target_names = dataset.target_names

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X = vectorizer.fit_transform(texts)

X.shape, y.shape, target_names

## 2. Evaluation helpers

In [ ]:
def evaluate_subset(X_subset, y_subset, label, wheel="baseline", subset_type="baseline", trial=-1):
    """Train/test evaluate a subset with stratified split."""
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset,
        y_subset,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y_subset,
    )

    clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    pred = clf.predict(X_test)
    return {
        "label": label,
        "wheel": wheel,
        "subset_type": subset_type,
        "trial": trial,
        "n_samples": X_subset.shape[0],
        "n_features": X_subset.shape[1],
        "nnz": X_subset.nnz,
        "fit_time_seconds": fit_time,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    }


def wheel_row_ids(n_rows, wheel_name):
    row_ids = np.arange(n_rows)
    return np.array(pre_oracle_candidates_by_name(row_ids, wheel_name), dtype=int)


def random_row_ids(n_rows, n_keep, seed):
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(np.arange(n_rows), size=n_keep, replace=False))


evaluate_subset(X, y, "baseline_all_rows")

## 3. Run matched wheel vs random trials

For each wheel, use the exact wheel-retained sample count, then sample random matched subsets of the same size.

In [ ]:
results = []

# Baseline on all rows.
results.append(evaluate_subset(X, y, "baseline_all_rows", wheel="baseline", subset_type="baseline", trial=-1))

for wheel_name in STANDARD_WHEELS:
    ids = wheel_row_ids(X.shape[0], wheel_name)
    Xw, yw = X[ids], y[ids]
    n_keep = len(ids)

    # Deterministic wheel subset.
    results.append(evaluate_subset(Xw, yw, f"{wheel_name}_wheel", wheel=wheel_name, subset_type="wheel", trial=-1))

    # Random matched subsets.
    for trial in range(N_RANDOM_TRIALS):
        rid = random_row_ids(X.shape[0], n_keep, seed=RANDOM_STATE + 1000 * (trial + 1) + len(wheel_name))
        Xr, yr = X[rid], y[rid]
        results.append(evaluate_subset(Xr, yr, f"{wheel_name}_random_{trial:02d}", wheel=wheel_name, subset_type="random_matched", trial=trial))

results_df = pd.DataFrame(results)
results_df

## 4. Summarize random distributions and wheel deltas

In [ ]:
random_summary = (
    results_df[results_df["subset_type"] == "random_matched"]
    .groupby("wheel")
    .agg(
        n_samples=("n_samples", "first"),
        random_bal_acc_mean=("balanced_accuracy", "mean"),
        random_bal_acc_std=("balanced_accuracy", "std"),
        random_acc_mean=("accuracy", "mean"),
        random_acc_std=("accuracy", "std"),
        random_fit_time_mean=("fit_time_seconds", "mean"),
        random_fit_time_std=("fit_time_seconds", "std"),
    )
    .reset_index()
)

wheel_results = results_df[results_df["subset_type"] == "wheel"][[
    "wheel", "n_samples", "balanced_accuracy", "accuracy", "fit_time_seconds"
]].rename(columns={
    "balanced_accuracy": "wheel_bal_acc",
    "accuracy": "wheel_acc",
    "fit_time_seconds": "wheel_fit_time",
})

delta_summary = wheel_results.merge(random_summary, on=["wheel", "n_samples"])
delta_summary["delta_bal_acc_vs_random_mean"] = delta_summary["wheel_bal_acc"] - delta_summary["random_bal_acc_mean"]
delta_summary["delta_acc_vs_random_mean"] = delta_summary["wheel_acc"] - delta_summary["random_acc_mean"]
delta_summary["delta_fit_time_vs_random_mean"] = delta_summary["wheel_fit_time"] - delta_summary["random_fit_time_mean"]

results_csv = data_dir / "05_modwheel_vs_random_trials.csv"
summary_csv = data_dir / "05_modwheel_vs_random_summary.csv"
results_df.to_csv(results_csv, index=False)
delta_summary.to_csv(summary_csv, index=False)

delta_summary

## 5. Figure 1 — Balanced accuracy: wheel vs random matched

In [ ]:
order = ["mod30", "mod210", "mod2310"]
random_data = [
    results_df[(results_df["wheel"] == w) & (results_df["subset_type"] == "random_matched")]["balanced_accuracy"].values
    for w in order
]
wheel_points = [
    results_df[(results_df["wheel"] == w) & (results_df["subset_type"] == "wheel")]["balanced_accuracy"].iloc[0]
    for w in order
]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
positions = np.arange(1, len(order) + 1)
ax.boxplot(random_data, positions=positions, widths=0.5, labels=order)
ax.scatter(positions, wheel_points, marker="D", s=70, label="wheel subset")
ax.set_title("20news Balanced Accuracy: Wheel vs Random Matched Subsampling")
ax.set_xlabel("Wheel / matched retained count")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, axis="y", alpha=0.35)
ax.legend()
fig.tight_layout()

fig1_svg = fig_dir / "figure_05a_balanced_accuracy_wheel_vs_random.svg"
fig1_png = fig_dir / "figure_05a_balanced_accuracy_wheel_vs_random.png"
fig.savefig(fig1_svg)
fig.savefig(fig1_png, dpi=220)
plt.show()
fig1_svg, fig1_png

## 6. Figure 2 — Training time: wheel vs random matched

In [ ]:
random_time_data = [
    results_df[(results_df["wheel"] == w) & (results_df["subset_type"] == "random_matched")]["fit_time_seconds"].values
    for w in order
]
wheel_time_points = [
    results_df[(results_df["wheel"] == w) & (results_df["subset_type"] == "wheel")]["fit_time_seconds"].iloc[0]
    for w in order
]

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.boxplot(random_time_data, positions=positions, widths=0.5, labels=order)
ax.scatter(positions, wheel_time_points, marker="D", s=70, label="wheel subset")
ax.set_title("20news Training-Time Proxy: Wheel vs Random Matched Subsampling")
ax.set_xlabel("Wheel / matched retained count")
ax.set_ylabel("Training time (seconds)")
ax.grid(True, axis="y", alpha=0.35)
ax.legend()
fig.tight_layout()

fig2_svg = fig_dir / "figure_05b_fit_time_wheel_vs_random.svg"
fig2_png = fig_dir / "figure_05b_fit_time_wheel_vs_random.png"
fig.savefig(fig2_svg)
fig.savefig(fig2_png, dpi=220)
plt.show()
fig2_svg, fig2_png

## 7. Figure 3 — Delta from random matched mean

In [ ]:
plot_delta = delta_summary.set_index("wheel").loc[order].reset_index()

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(plot_delta["wheel"], plot_delta["delta_bal_acc_vs_random_mean"])
ax.set_title("Wheel Balanced Accuracy Minus Random Matched Mean")
ax.set_xlabel("Wheel")
ax.set_ylabel("Δ balanced accuracy")
ax.grid(True, axis="y", alpha=0.35)
for i, value in enumerate(plot_delta["delta_bal_acc_vs_random_mean"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")
fig.tight_layout()

fig3_svg = fig_dir / "figure_05c_delta_balanced_accuracy_vs_random.svg"
fig3_png = fig_dir / "figure_05c_delta_balanced_accuracy_vs_random.png"
fig.savefig(fig3_svg)
fig.savefig(fig3_png, dpi=220)
plt.show()
fig3_svg, fig3_png

## 8. Interpretation helper

In [ ]:
for _, row in delta_summary.iterrows():
    wheel = row["wheel"]
    delta = row["delta_bal_acc_vs_random_mean"]
    print(f"{wheel}: wheel balanced accuracy - random mean = {delta:+.4f}")

print("""
Interpretation rules:

- If wheel ≈ random mean: deterministic wheel filtering provides reproducible reduction comparable to random subsampling.
- If wheel > random mean: wheel filtering may preserve useful structure beyond random matched subsampling.
- If wheel < random mean: wheel filtering still reduces input size, but row-ID bias may matter for this dataset.

Guardrail:
This is a row-ID adapter experiment, not a proof of QOS improvement or quantum advantage.
""")

## 9. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/05_modwheel_vs_random_trials.csv",
    "data/05_modwheel_vs_random_summary.csv",
    "figures/figure_05a_balanced_accuracy_wheel_vs_random.svg",
    "figures/figure_05b_fit_time_wheel_vs_random.svg",
    "figures/figure_05c_delta_balanced_accuracy_vs_random.svg",
]:
    files.download(path)